<a href="https://colab.research.google.com/github/Durgesh-Kumar-Dewangan/TestAing-Solutions-Pvt-Ltd-aiensured.com--2026/blob/main/working_Multi_agentic_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU langchain-google-genai langgraph faiss-cpu sentence-transformers langchain-community langchain-huggingface

In [ ]:
# 1. Installation
!pip install -q -U crewai crewai[tools] google-genai sentence-transformers faiss-cpu pymupdf gradio tenacity tqdm

print("✅ Packages installed.")

✅ Packages installed.


In [ ]:
# ⚙️ Installation Cell
# Run this first

!pip install -q -U \
    crewai \
    crewai[tools] \
    sentence-transformers \
    faiss-cpu \
    langchain-community \
    gradio \
    pymupdf \
    tenacity \
    tqdm \
    numpy \
    websockets

print("✅ All packages installed successfully!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.34.1 which is incompatible.
google-adk 1.29.0 requires pydantic<3.0.0,>=2.12.0, but you have pydantic 2.11.10 which is incompatible.
google-adk 1.29.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 16.0 which is incompatible.
numba 0.

In [ ]:
# 2. API Key
import os
from google.colab import userdata
from getpass import getpass

if not os.getenv("GEMINI_API_KEY"):
    try:
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
        print(" API key loaded from Secrets.")
    except:
        os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API Key: ")

print(" Gemini API Key configured.")

 Gemini API Key configured.


In [ ]:
# ============================
# 3. PDF UPLOAD & PROCESSING
# ============================

import fitz
from typing import List
from tqdm.notebook import tqdm

documents = []

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    return "".join(page.get_text("text") for page in doc).strip()

def chunk_text(text: str, chunk_size=700, overlap=80):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start = end - overlap
    return [c for c in chunks if c]

print("✅ PDF functions ready.")

✅ PDF functions ready.


In [ ]:
from google.colab import files

print("📤 Please upload your PDF file(s)")
uploaded = files.upload()

all_chunks = []
for fn in uploaded.keys():
    if fn.lower().endswith('.pdf'):
        print(f"Processing {fn}...")
        text = extract_text_from_pdf(fn)
        chunks = chunk_text(text)
        all_chunks.extend(chunks)
        print(f"✓ {len(chunks)} chunks from {fn}")

documents = all_chunks
print(f"\n🎉 Total chunks: {len(documents)}")

📤 Please upload your PDF file(s)


Saving RAG_Research_Paper - Copy (1).pdf to RAG_Research_Paper - Copy (1) (4).pdf
Processing RAG_Research_Paper - Copy (1) (4).pdf...
✓ 177 chunks from RAG_Research_Paper - Copy (1) (4).pdf

🎉 Total chunks: 177


In [ ]:
# 4. Embeddings + FAISS
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding model loaded.")

embeddings = embedding_model.encode(documents, show_progress_bar=True).astype('float32')
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print(f"✅ FAISS index ready with {len(documents)} chunks.")

def retrieve_documents(query: str, k=5):
    q_emb = embedding_model.encode([query]).astype('float32')
    faiss.normalize_L2(q_emb)
    _, idx = index.search(q_emb, k)
    return [documents[i] for i in idx[0]]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded.


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

✅ FAISS index ready with 177 chunks.


In [ ]:
# ============================
# 5. RAG TOOL
# ============================
# 5. RAG Tool
from crewai.tools import BaseTool

class PDFRAGTool(BaseTool):
    name: str = "PDF_Retriever"
    description: str = "Retrieves relevant text from uploaded PDF documents."

    def _run(self, query: str) -> str:
        docs = retrieve_documents(query, k=6)
        return "Retrieved from PDFs:\n" + "\n\n".join([f"Chunk {i+1}: {d[:600]}..." for i, d in enumerate(docs)])

rag_tool = PDFRAGTool()

In [ ]:
# ============================
# 6. AGENTS & LLM
# ============================
# 6. Agents with Stable LLM
from crewai import Agent, Task, Crew, Process
from crewai.llm import LLM
from tenacity import retry, stop_after_attempt, wait_exponential

llm = LLM(
    model="gemini/gemini-1.5-flash",   # Most stable for free tier
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.2,
    max_tokens=1500,
)

planner = Agent(role="Planner", goal="Create simple plan", backstory="Efficient planner", llm=llm, verbose=False)
retriever = Agent(role="Retriever", goal="Get relevant PDF content", backstory="Retrieval expert", llm=llm, tools=[rag_tool], verbose=False)
analyst = Agent(role="Analyst", goal="Extract insights", backstory="Logical analyst", llm=llm, verbose=False)
critic = Agent(role="Critic", goal="Validate and improve answer", backstory="Quality checker", llm=llm, verbose=False)
generator = Agent(role="Generator", goal="Write clear final answer", backstory="Professional writer", llm=llm, verbose=False)

In [ ]:
# ============================
# 7. MAIN MULTI-AGENT FUNCTION
# ============================
# 7. Main Function with Stronger Retry
@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=12, min=15, max=90),
    reraise=True
)
def run_multi_agent_rag(query: str):
    if not documents:
        return "Please upload at least one PDF first."

    print(f"\n🔍 Query: {query}")

    tasks = [
        Task(description=f"Plan: {query}", expected_output="Short plan", agent=planner),
        Task(description="Retrieve relevant content from PDFs", expected_output="Context", agent=retriever),
        Task(description="Analyze the retrieved content", expected_output="Key insights", agent=analyst),
        Task(description="Write a draft answer", expected_output="Draft", agent=generator),
        Task(description="Review, fix issues, and give final answer with confidence score (0-100)",
             expected_output="Final answer with confidence", agent=critic)
    ]

    crew = Crew(
        agents=[planner, retriever, analyst, generator, critic],
        tasks=tasks,
        process=Process.sequential,
        verbose=1,
        memory=False,
        cache=True
    )

    return crew.kickoff()

In [ ]:
# ============================
# 8. DEMO
# ============================

if documents:
    test_query = "Summarize the main findings or key points from the uploaded document."
    print("🚀 Running Demo...\n")
    answer = run_multi_agent_rag(test_query)
    print("\n" + "="*80)
    print("📝 FINAL ANSWER:")
    print(answer)
else:
    print("⚠️  Please upload PDFs first before running demo.")

🚀 Running Demo...


🔍 Query: Summarize the main findings or key points from the uploaded document.


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 92632d9f-403e-4d46-98fd-1600a2378b05                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Plan: Summarize the main findings or key points from the uploaded document.                              │
│  ID: d77acb21-fe1c-4c18-9692-f33934cb0313                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Planner                                                                                                 │
│                                                                                                                 │
│  Task: Plan: Summarize the main findings or key points from the uploaded document.                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:Google Gemini API error: 404 - models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.


...

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: Google Gemini API error: 404 - models/gemini-1.5-flash is not found for API version v1beta, or is not   │
│  supported for generateContent. Call ListModels to see the list of available models and their supported         │
│  methods.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Plan: Summarize the main findings or key points from the uploaded document.                              │
│  Agent: Planner                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 8a87fbf9-2d54-417e-9bb5-dc99e45a6084                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [ ]:
# 9. Gradio UI
import gradio as gr

def chat(query):
    if not query.strip():
        return "Please ask a question."
    try:
        return run_multi_agent_rag(query)
    except Exception as e:
        return f"Error: {str(e)}\n\nTip: Wait 30-60 seconds and try again if you see 503 error."

gr.Interface(
    fn=chat,
    inputs=gr.Textbox(lines=3, placeholder="Ask about your PDF..."),
    outputs=gr.Markdown(),
    title="Multi-Agent RAG with PDF Upload (Stable)",
    description="Using Gemini 1.5 Flash • 5 Agents • Fixed for 503 errors",
    examples=[
        ["Summarize the key findings"],
        ["What are the main conclusions?"],
        ["Extract important recommendations"]
    ]
).launch(share=False)